# Task A — TAPT + 10 epochs + two-layer reinitialization

This is the first combined Task A recipe that puts the three strongest measured component changes together: domain-adaptive pretraining (TAPT), ten fine-tuning epochs, and reinitialization of the top two MuRIL encoder layers. It evaluates the MuRIL arm with five-fold out-of-fold predictions on the original labels plus the released labelled validation rows, then compares it with a matched demojized TF-IDF/SVM arm and a searched ensemble.

The released validation labels are included because they are now available locally. This is an evaluation diagnostic, not a CodaBench submission notebook. Expected runtime is roughly 6–10 hours on a Kaggle T4 ×2, plus the TAPT pass.

In [ ]:
import os, pathlib, shutil, subprocess, sys
WORK = '/kaggle/working/hastika'
if os.path.isdir(WORK + '/.git'):
    subprocess.run(['git', '-C', WORK, 'pull', '--ff-only', 'origin', 'task-b'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', 'task-b', '--depth', '1', 'https://github.com/robinpnalex/Hastika-ICON2026.git', WORK], check=True)
os.chdir(WORK)
os.environ['PYTHONPATH'] = os.path.join(WORK, 'src')
sys.path.insert(0, os.path.join(WORK, 'src'))
pathlib.Path('artifacts/logs').mkdir(parents=True, exist_ok=True)
from urllib.request import urlretrieve
raw_base = 'https://raw.githubusercontent.com/robinpnalex/Hastika-ICON2026/task-b/'
for rel in ['data/raw/hastika_binary_validation.csv']:
    p = pathlib.Path(rel)
    if not p.exists():
        p.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(raw_base + rel, p)
    assert p.exists() and p.stat().st_size > 0, f'missing required data: {rel}'
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import numpy as np, pandas as pd, torch
from sklearn.metrics import accuracy_score, f1_score
from hastika.common.preprocessing import dedupe_index
assert torch.cuda.is_available(), 'enable a GPU'
print('gpu:', torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print('$', ' '.join(map(str, cmd)), flush=True)
    fh = open(log, 'w') if log else None
    p = subprocess.Popen([str(x) for x in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f'exit {p.returncode}: {cmd}')


## 1. Stage the combined labelled corpus

In [ ]:
from pathlib import Path
train_path = Path('data/raw/binary_train.csv')
val_path = Path('data/raw/hastika_binary_validation.csv')
combined = pd.concat([pd.read_csv(train_path), pd.read_csv(val_path)], ignore_index=True)
combined = combined.drop_duplicates('id', keep='first').reset_index(drop=True)
assert len(combined) == 7252 and combined['id'].is_unique
combined_path = Path('data/derived/task_a_run24_combined.csv')
combined_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(combined_path, index=False)
CANON_TRAIN = Path('data/raw/binary_train.csv')
BACKUP_TRAIN = Path('data/raw/binary_train.csv.before_run24')
assert not BACKUP_TRAIN.exists(), 'stale backup exists; stop and inspect before rerunning'
shutil.copy2(CANON_TRAIN, BACKUP_TRAIN)
shutil.copy2(combined_path, CANON_TRAIN)
print('staged rows:', len(combined))


## 2. Build the Task-A TAPT checkpoint

TAPT reads text only. It uses the staged labelled corpus as an in-domain language-model corpus, plus the permitted external Kannada text, and reads no labels. The validation comments are included because this notebook is explicitly a combined-corpus diagnostic; the choice is recorded in the log.

In [ ]:
TAPT_TAG = 'tapt-task-a-run24'
tapt_out = Path('artifacts/runs') / TAPT_TAG
if not (tapt_out / 'config.json').exists():
    run([sys.executable, '-u', '-m', 'hastika.task_b.tapt',
         '--corpus', 'data/raw/binary_train.csv', 'data/external/offenseval_kn.csv',
         '--allow-transductive', '--out', str(tapt_out), '--val-frac', '0',
         '--min-words', '1', '--no-dedupe', '--epochs', '8'],
        log='artifacts/logs/task_a_run24_tapt.log')
else:
    print('reusing existing TAPT checkpoint:', tapt_out)


## 3. Generate matched five-fold OOF probabilities

In [ ]:
SVM_TAG = 'a_run24_svm'
MURIL_TAG = 'a_run24_tapt10_reinit2'
run([sys.executable, '-u', '-m', 'hastika.models.baseline_svm', '--task', 'a', '--tag', SVM_TAG, '--demojize'], log=f'artifacts/logs/{SVM_TAG}.log')
common = [sys.executable, '-u', '-m', 'hastika.models.muril', '--model', str(tapt_out), '--tag', MURIL_TAG, '--folds', '5', '--seeds', '42', '--epochs', '10', '--bs', '8', '--grad-accum', '2', '--eval-bs', '32', '--select', 'last', '--reinit-layers', '2']
run(common, log=f'artifacts/logs/{MURIL_TAG}.log')
run([sys.executable, '-u', '-m', 'hastika.models.ensemble', '--task', 'a', '--tags', SVM_TAG, MURIL_TAG, '--out', 'a_run24_ensemble'], log='artifacts/logs/a_run24_ensemble.log')
print('Run 24 OOF and nested ensemble evaluation completed')


## 4. Save a compact report

In [ ]:
keep = dedupe_index(combined['Comment'].tolist(), combined['Label'].tolist(), 'task A')
fit_df = combined.iloc[keep].reset_index(drop=True)
y = (fit_df['Label'] == 'Hate').astype(int).to_numpy()
report = []
for tag in [SVM_TAG, MURIL_TAG]:
    p = np.load(Path('artifacts/runs') / tag / 'oof_probs.npy')[:, 1]
    pred = (p > 0.5).astype(int)
    report.append({'model': tag, 'macro_f1': f1_score(y, pred, average='macro'), 'accuracy': accuracy_score(y, pred)})
OUT = Path('/kaggle/working/task_a_run24_outputs')
OUT.mkdir(parents=True, exist_ok=True)
pd.DataFrame(report).to_csv(OUT / 'individual_oof_summary.csv', index=False)
print(pd.DataFrame(report).to_string(index=False))
print('The ensemble command above prints the searched blend and nested estimate.')
print('DOWNLOAD LOGS/PROBABILITIES FROM:', OUT, 'and artifacts/runs')


## 5. Restore the canonical training file

In [ ]:
shutil.move(str(BACKUP_TRAIN), str(CANON_TRAIN))
print('restored', CANON_TRAIN)
